Đọc dữ liệu và làm schaj

In [5]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import (
    train_test_split,
    ParameterSampler
)
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
df = pd.read_csv("../../data/raw/Diamonds Prices2022.csv")
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

# Loại bỏ kích thước không hợp lệ
df = df[
    (df["x"] > 0) &
    (df["y"] > 0) &
    (df["z"] > 0)
]

# Xóa dòng trùng
df = df.drop_duplicates().copy()

# Mã hóa thuộc tính phân loại
categorical_cols = ["cut", "color", "clarity"]

encoder = OrdinalEncoder()
df[categorical_cols] = encoder.fit_transform(
    df[categorical_cols]
)

X = df.drop(columns=["price"])
y = df["price"]

print("Số dòng:", len(df))
print("Số thuộc tính đầu vào:", X.shape[1])
print("Số dòng thiếu dữ liệu:", df.isnull().sum().sum())


Số dòng: 53775
Số thuộc tính đầu vào: 9
Số dòng thiếu dữ liệu: 0


Chia Train, Validation và Test

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_subtrain, X_val, y_subtrain, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42
)

print("Train:", len(X_train))
print("Subtrain:", len(X_subtrain))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Train: 43020
Subtrain: 34416
Validation: 8604
Test: 10755


Randomized Search và huấn luyện

In [8]:
param_grid = {
    "n_estimators": [100, 150, 200, 250],
    "max_depth": [10, 15, 20, 25, None],
    "min_samples_leaf": [1, 2, 3],
    "min_samples_split": [2, 5, 10],
    "max_features": [1.0, 0.8, 0.6]
}

param_list = list(ParameterSampler(
    param_grid,
    n_iter=20,
    random_state=42
))

best_rmse = float("inf")
best_params = None
search_results = []

start_time = time.time()

for i, params in enumerate(param_list, start=1):

    model = RandomForestRegressor(
        **params,
        criterion="squared_error",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_subtrain, y_subtrain)

    y_val_pred = model.predict(X_val)
    rmse_val = np.sqrt(
        mean_squared_error(y_val, y_val_pred)
    )

    search_results.append({
        **params,
        "RMSE Validation": rmse_val
    })

    if rmse_val < best_rmse:
        best_rmse = rmse_val
        best_params = params.copy()

    print(
        f"Đã thử {i}/20 cấu hình | "
        f"RMSE Validation: {rmse_val:.4f}",
        flush=True
    )

search_time = time.time() - start_time

print("\nTham số tốt nhất:", best_params)
print(f"RMSE Validation tốt nhất: {best_rmse:.4f}")
print(f"Thời gian tìm kiếm: {search_time:.2f} giây")

Đã thử 1/20 cấu hình | RMSE Validation: 580.5823
Đã thử 2/20 cấu hình | RMSE Validation: 623.0848
Đã thử 3/20 cấu hình | RMSE Validation: 585.4729
Đã thử 4/20 cấu hình | RMSE Validation: 622.8593
Đã thử 5/20 cấu hình | RMSE Validation: 575.3933
Đã thử 6/20 cấu hình | RMSE Validation: 623.0122
Đã thử 7/20 cấu hình | RMSE Validation: 584.8489
Đã thử 8/20 cấu hình | RMSE Validation: 585.8093
Đã thử 9/20 cấu hình | RMSE Validation: 579.7485
Đã thử 10/20 cấu hình | RMSE Validation: 576.8436
Đã thử 11/20 cấu hình | RMSE Validation: 591.9342
Đã thử 12/20 cấu hình | RMSE Validation: 572.3219
Đã thử 13/20 cấu hình | RMSE Validation: 580.9417
Đã thử 14/20 cấu hình | RMSE Validation: 622.9673
Đã thử 15/20 cấu hình | RMSE Validation: 579.0138
Đã thử 16/20 cấu hình | RMSE Validation: 581.2280
Đã thử 17/20 cấu hình | RMSE Validation: 580.4179
Đã thử 18/20 cấu hình | RMSE Validation: 627.7853
Đã thử 19/20 cấu hình | RMSE Validation: 587.2289
Đã thử 20/20 cấu hình | RMSE Validation: 598.9717

Tham số 

Huấn luyện mô hình tốt nhất và đánh giá

In [9]:
start_fit = time.time()

best_model = RandomForestRegressor(
    **best_params,
    criterion="squared_error",
    random_state=42,
    n_jobs=-1
)

best_model.fit(X_train, y_train)

fit_time = time.time() - start_fit

y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred)
    }

train_metrics = evaluate(y_train, y_train_pred)
test_metrics = evaluate(y_test, y_test_pred)

results_holdout = pd.DataFrame(
    [train_metrics, test_metrics],
    index=["Train", "Test"]
)

print("KẾT QUẢ RANDOM FOREST HOLD-OUT MỨC 5")
print(results_holdout.to_string())

print("\nTham số tốt nhất:", best_params)
print(f"Thời gian tìm kiếm: {search_time:.2f} giây")
print(f"Thời gian huấn luyện cuối: {fit_time:.2f} giây")
print(f"Tổng thời gian: {search_time + fit_time:.2f} giây")

KẾT QUẢ RANDOM FOREST HOLD-OUT MỨC 5
              MAE            MSE        RMSE        R2
Train  189.681011  136928.755516  370.038857  0.991393
Test   273.064331  281459.715036  530.527770  0.982190

Tham số tốt nhất: {'n_estimators': 150, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 1.0, 'max_depth': 15}
Thời gian tìm kiếm: 44.69 giây
Thời gian huấn luyện cuối: 2.08 giây
Tổng thời gian: 46.77 giây


Lưu KQ

In [10]:
import os

os.makedirs("results", exist_ok=True)

results_holdout.to_csv(
    "results/RF_Level5_Holdout.csv"
)

pd.DataFrame(search_results).sort_values(
    "RMSE Validation"
).to_csv(
    "results/RF_Level5_Holdout_Search.csv",
    index=False
)

print("Đã lưu kết quả Hold-out mức 5.")

Đã lưu kết quả Hold-out mức 5.
